# 🏥 Improved Hospital Readmission Prediction Pipeline
### Leakage-Free Preprocessing, Clinical Feature Engineering, Cross-Validation, Class Imbalance Optimization & Multi-Model Benchmarking

This notebook implements an advanced, leak-free machine learning pipeline for predicting 30-day hospital readmissions on the **Diabetes 130-US Hospitals (1999–2008)** dataset.

---
### Key Enhancements over Baseline:
1. **Zero Data Leakage:** Group-aware splitting (GroupShuffleSplit on patient_nbr) ensures no patient appears across splits. All scalers and encoders are strictly fitted on training data.
2. **Clinical Feature Engineering:**
   - Grouping 2,200+ sparse ICD-9 codes into 9 established clinical categories (Strack et al., 2014).
   - Utilization intensity ratios (inpatient ratio, emergency ratio, prior visit intensity).
   - Clinical complexity composite and daily procedure/medication rates.
   - Polypharmacy counts, medication titration indicators, and glycemic testing flags.
3. **Algorithms Evaluated:** Logistic Regression, Decision Tree, Random Forest, and XGBoost.
4. **5-Fold Stratified Group Cross-Validation:** Hyperparameter tuning with StratifiedGroupKFold on patient clusters.
5. **Class Imbalance Strategy:** Empirical comparison of Cost-Sensitive Class Weighting vs SMOTE oversampling.
6. **Out-of-Fold Threshold Optimization:** Calibrated decision thresholds via Precision-Recall curve analysis.
7. **Comprehensive Metrics:** Accuracy, Precision, Recall, F1-Score, ROC-AUC, and PR-AUC on held-out test encounters.


In [ ]:
# 1. Imports and Environment Setup
import os
import sys
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Libraries loaded successfully.')


## 2. Domain-Informed Clinical Feature Engineering
We transform raw tabular codes into high-signal clinical indicators:
- **ICD-9 Diagnosis Mapping:** Maps diagnosis codes to Circulatory, Respiratory, Digestive, Diabetes, Injury, Musculoskeletal, Genitourinary, Neoplasms, and Other.
- **Utilization Ratios:** Inpatient/Emergency visit ratios relative to total healthcare utilization.
- **Complexity Scores:** Combining procedures, lab tests, medications, and diagnosis counts.


In [ ]:
def map_icd9_clinical_group(val):
    if pd.isna(val) or str(val).strip() in ['?', '', 'None', 'nan']:
        return 'Missing'
    s = str(val).strip()
    if s.startswith(('V', 'v')): return 'Supplementary_V'
    if s.startswith(('E', 'e')): return 'External_E'
    try:
        code = float(s)
        if (390 <= code <= 459) or code == 785: return 'Circulatory'
        elif (460 <= code <= 519) or code == 786: return 'Respiratory'
        elif (520 <= code <= 579) or code == 787: return 'Digestive'
        elif int(code) == 250: return 'Diabetes'
        elif 800 <= code <= 999: return 'Injury'
        elif 710 <= code <= 739: return 'Musculoskeletal'
        elif (580 <= code <= 629) or code == 788: return 'Genitourinary'
        elif 140 <= code <= 239: return 'Neoplasms'
        else: return 'Other'
    except ValueError:
        return 'Other'

def engineer_patient_features(raw_df):
    df = raw_df.copy()
    df['target_readmitted_30d'] = (df['readmitted'] == '<30').astype(int)
    
    age_map = {'[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, '[40-50)': 45,
               '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, '[80-90)': 85, '[90-100)': 95}
    df['age_midpoint'] = df['age'].map(age_map).fillna(65).astype(float)
    df['is_senior'] = (df['age_midpoint'] >= 65).astype(int)
    
    for diag_col in ['diag_1', 'diag_2', 'diag_3']:
        df[f'{diag_col}_group'] = df[diag_col].apply(map_icd9_clinical_group)
        
    for cat in ['Diabetes', 'Circulatory', 'Respiratory', 'Genitourinary', 'Neoplasms', 'Digestive', 'Injury', 'Musculoskeletal']:
        df[f'has_diag_{cat.lower()}'] = ((df['diag_1_group'] == cat) | (df['diag_2_group'] == cat) | (df['diag_3_group'] == cat)).astype(int)
        
    df['total_prior_visits'] = df['number_inpatient'] + df['number_emergency'] + df['number_outpatient']
    df['has_prior_inpatient'] = (df['number_inpatient'] > 0).astype(int)
    df['has_prior_emergency'] = (df['number_emergency'] > 0).astype(int)
    df['has_prior_outpatient'] = (df['number_outpatient'] > 0).astype(int)
    df['inpatient_visit_ratio'] = df['number_inpatient'] / (df['total_prior_visits'] + 1.0)
    df['emergency_visit_ratio'] = df['number_emergency'] / (df['total_prior_visits'] + 1.0)
    df['prior_visit_intensity'] = df['total_prior_visits'] / (df['time_in_hospital'] + 1.0)
    
    df['lab_procedures_per_day'] = df['num_lab_procedures'] / df['time_in_hospital']
    df['procedures_per_day'] = df['num_procedures'] / df['time_in_hospital']
    df['medications_per_day'] = df['num_medications'] / df['time_in_hospital']
    df['clinical_complexity'] = df['num_procedures'] + (df['num_lab_procedures'] / 10.0) + (df['num_medications'] / 5.0) + df['number_diagnoses']
    
    all_meds = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
    avail_meds = [m for m in all_meds if m in df.columns]
    df['num_active_meds'] = (df[avail_meds] != 'No').sum(axis=1)
    df['num_meds_up'] = (df[avail_meds] == 'Up').sum(axis=1)
    df['num_meds_down'] = (df[avail_meds] == 'Down').sum(axis=1)
    df['num_meds_steady'] = (df[avail_meds] == 'Steady').sum(axis=1)
    df['has_insulin'] = (df['insulin'] != 'No').astype(int)
    df['has_metformin'] = (df['metformin'] != 'No').astype(int)
    df['med_changed_flag'] = (df['change'] == 'Ch').astype(int)
    df['diabetes_med_flag'] = (df['diabetesMed'] == 'Yes').astype(int)
    
    df['a1c_tested'] = (~df['A1Cresult'].isin(['None', 'Not Tested', '?'])).astype(int)
    df['a1c_high'] = (df['A1Cresult'].isin(['>7', '>8'])).astype(int)
    df['glucose_tested'] = (~df['max_glu_serum'].isin(['None', 'Not Tested', '?'])).astype(int)
    df['glucose_high'] = (df['max_glu_serum'].isin(['>200', '>300'])).astype(int)
    
    for col in ['race', 'payer_code']:
        df[col] = df[col].replace('?', 'Unknown').fillna('Unknown')
        
    top_specialties = ['InternalMedicine', 'Emergency/Trauma', 'Family/GeneralPractice', 'Cardiology', 'Surgery-General', 'Nephrology', 'Orthopedics', 'Orthopedics-Reconstructive', 'Radiologist', 'Pulmonology']
    df['specialty_grouped'] = df['medical_specialty'].apply(lambda x: x if x in top_specialties else ('Missing' if x in ['?', 'Unknown', np.nan] else 'Other_Specialty'))
    
    for col in ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']:
        df[col] = df[col].astype(str)
        
    for col in ['max_glu_serum', 'A1Cresult']:
        df[col] = df[col].replace('None', 'Not Tested').fillna('Not Tested')
        
    return df


## 3. Preprocessing Pipeline & Patient-Separated Split
We partition the dataset with GroupShuffleSplit on patient_nbr so that no patient's encounters appear in both train and test partitions.


In [ ]:
NUMERICAL_FEATURES = [
    'age_midpoint', 'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient',
    'number_diagnoses', 'total_prior_visits', 'inpatient_visit_ratio',
    'emergency_visit_ratio', 'prior_visit_intensity', 'lab_procedures_per_day',
    'procedures_per_day', 'medications_per_day', 'clinical_complexity',
    'num_active_meds', 'num_meds_up', 'num_meds_down', 'num_meds_steady'
]

BINARY_FEATURES = [
    'is_senior', 'has_prior_inpatient', 'has_prior_emergency', 'has_prior_outpatient',
    'has_diag_diabetes', 'has_diag_circulatory', 'has_diag_respiratory',
    'has_diag_genitourinary', 'has_diag_neoplasms', 'has_diag_digestive',
    'has_diag_injury', 'has_diag_musculoskeletal', 'has_insulin', 'has_metformin',
    'med_changed_flag', 'diabetes_med_flag', 'a1c_tested', 'a1c_high',
    'glucose_tested', 'glucose_high'
]

CATEGORICAL_FEATURES = [
    'race', 'gender', 'admission_type_id', 'discharge_disposition_id',
    'admission_source_id', 'payer_code', 'specialty_grouped',
    'diag_1_group', 'diag_2_group', 'diag_3_group',
    'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed',
    'insulin', 'metformin', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'glimepiride'
]

ALL_INPUT_FEATURES = NUMERICAL_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERICAL_FEATURES),
        ('bin', 'passthrough', BINARY_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES)
    ]
)

raw_data = pd.read_csv('../datasets/dataset1/diabetic_data.csv')
processed_df = engineer_patient_features(raw_data)

X_df = processed_df[ALL_INPUT_FEATURES].copy()
y = processed_df['target_readmitted_30d'].values
patient_ids = processed_df['patient_nbr'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X_df, y, groups=patient_ids))

X_train_df, X_test_df = X_df.iloc[train_idx].copy(), X_df.iloc[test_idx].copy()
y_train, y_test = y[train_idx], y[test_idx]
train_patients, test_patients = patient_ids[train_idx], patient_ids[test_idx]

# Strictly fit preprocessor on train data only
X_train_processed = preprocessor.fit_transform(X_train_df)
X_test_processed = preprocessor.transform(X_test_df)

print(f'Train samples: {len(X_train_processed):,}, Test samples: {len(X_test_processed):,}')
print(f'Engineered features: {X_train_processed.shape[1]} (vs 2,344 in baseline)')


## 4. Stratified Group Cross-Validation & Hyperparameter Tuning
We perform 5-fold cross-validation using StratifiedGroupKFold on the training set, tuning:
- **Logistic Regression:** Regularization strength C
- **Decision Tree:** max_depth and min_samples_leaf
- **Random Forest:** Depth, leaf size, and ensemble size
- **XGBoost:** Trees, depth, learning rate, subsampling, and feature subsampling


In [ ]:
# Evaluate candidates and retrieve saved evaluation results
eval_df = pd.read_csv('../results/improved_model_comparison.csv')
display(eval_df)


## 5. Class Imbalance Strategy Comparison: Cost-Sensitive Weights vs SMOTE
On high-dimensional tabular data with mixed continuous and discrete features, SMOTE interpolates points across Euclidean space, generating synthetic feature combinations that can distort discrete categories.


In [ ]:
smote_comparison = pd.read_csv('../results/smote_vs_class_weights.csv')
display(smote_comparison)


## 6. Precision-Recall Threshold Optimization
Using out-of-fold probability distributions, we select the threshold that optimizes the F1-score and evaluate precision-recall trade-offs.


In [ ]:
from IPython.display import Image
Image('../results/threshold_optimization_curves.png')


## 7. Model Evaluation on Untouched Test Set (20,153 Encounters)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ROC Curves
from sklearn.metrics import roc_curve, precision_recall_curve

models = {
    'Logistic Regression': joblib.load('../models/improved/logistic_regression_model.pkl'),
    'Decision Tree': joblib.load('../models/improved/decision_tree_model.pkl'),
    'Random Forest': joblib.load('../models/improved/random_forest_model.pkl'),
    'XGBoost': joblib.load('../models/improved/xgboost_model.pkl')
}

colors = {'Logistic Regression': '#2b5c8f', 'Decision Tree': '#d95f02', 'Random Forest': '#7570b3', 'XGBoost': '#1b9e77'}

for name, model in models.items():
    prob = model.predict_proba(X_test_processed)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    axes[0].plot(fpr, tpr, color=colors[name], lw=2.5, label=f'{name} (AUC={auc_val:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5)
axes[0].set_title('Test ROC Curves', fontsize=14, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

for name, model in models.items():
    prob = model.predict_proba(X_test_processed)[:, 1]
    p, r, _ = precision_recall_curve(y_test, prob)
    pr_auc = average_precision_score(y_test, prob)
    axes[1].plot(r, p, color=colors[name], lw=2.5, label=f'{name} (PR-AUC={pr_auc:.4f})')

axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', lw=1.5, label='No-Skill Baseline')
axes[1].set_title('Test Precision-Recall Curves', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()


## 8. Confusion Matrices at Optimal Thresholds


In [ ]:
Image('../results/confusion_matrices_improved.png')


## 9. Feature Importance Analysis


In [ ]:
Image('../results/feature_importance_xgb_improved.png')


## 10. Head-to-Head Comparison: Original Baseline vs Improved Pipeline


In [ ]:
consolidated_df = pd.read_csv('../results/full_baseline_vs_improved_comparison.csv')
display(consolidated_df)
Image('../results/baseline_vs_improved_comparison.png')
